# dist-send-recv-pair — ex1: implement broadcast via paired dist.send and dist.recv

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `dist-send-recv-pair`. Running the final beacon cell reports progress against the `Distributed: dist.send/recv pair` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: dist.send/recv pair` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dist-send-recv-pair`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dist-send-recv-pair"
DD_SUBTOPIC = "Distributed: dist.send/recv pair"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## torch.distributed quick refresher

PyTorch's collective-communication library (`torch.distributed`, aliased `dist`) lets multiple processes coordinate over tensors. The standard workflow:

1. **Each rank** runs the same function, parameterized by `rank` and `world_size`. Rank 0 is conventionally the 'driver'.
2. **`dist.init_process_group(backend=...)`** establishes the rendezvous. Backends:
   - `'nccl'` — NVIDIA's GPU-to-GPU primitive. Used in ARENA's multi-GPU setup. Requires CUDA + one process per GPU.
   - `'gloo'` — CPU-friendly. What you'll use in these drills (Colab CPU runtimes have no real GPUs).
3. **Pin a device** per rank: `torch.device(f'cuda:{rank}')` so each process owns exactly one GPU.
4. **Collective ops** (`all_reduce`, `broadcast`, `send`, `recv`) operate in-place on tensors of identical shape across all ranks.
5. **`dist.destroy_process_group()`** tears down at the end.

**Two ways to launch multiple ranks:**
- `torch.multiprocessing.spawn(fn, args=(...), nprocs=world_size)` — what ARENA uses. Spawn requires the worker fn be importable (not defined in `__main__`/a notebook cell).
- `mp.get_context('fork').Process(target=fn, args=...)` — Linux-only but works with cell-defined fns. The drills use this in tests so the worker can stay in the cell.

**Two-rank trick.** Colab gives ~2 CPU cores, so `world_size=2` is the right scale: enough to exercise the protocol, cheap enough to finish in seconds.

### This drill's atom: matched `send` / `recv`
`dist.send(tensor, dst=R)` on the sending rank pairs with `dist.recv(buffer, src=S)` on the receiving rank. Both calls BLOCK until the match completes. **Critical invariant:** the `tensor` on the sender and the `buffer` on the receiver must have the **same shape and dtype**, otherwise the receiver hangs (or in newer torch, errors with a confusing message).

The standard ARENA pattern (used in `broadcast`):
```python
if rank == src:
    for other in range(world_size):
        if other != src:
            dist.send(tensor, dst=other)
else:
    buf = t.zeros_like(tensor)
    dist.recv(buf, src=src)
    tensor.copy_(buf)
```
(The `tensor.copy_(buf)` step mutates the caller's tensor so the result is visible without rebinding.)

### Exercise 1 — implement broadcast via paired dist.send and dist.recv

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply matched `dist.send` / `dist.recv` (with a `torch.zeros_like` buffer + `tensor.copy_` writeback) to build a 1-source-to-N-rank broadcast on the `gloo` backend.
> Keywords: dist.send, dist.recv, broadcast, tensor.copy_, gloo
> ```

**KCs targeted:** `send-recv-shape-match`, `recv-into-zeros_like-buffer`

Implement `ex1_broadcast_worker(rank, world_size, port, src, payload, out_queue)`. Each worker:

1. Init `gloo` process group with `(rank, world_size, port)`.
2. Build the tensor: if `rank == src`, use `t.tensor(payload, dtype=t.float32)`; else, use `t.zeros(len(payload), dtype=t.float32)` (the receive buffer).
3. Run the matched send/recv broadcast:
   - If `rank == src`: loop `for other in range(world_size)` and `dist.send(tensor, dst=other)` for every other rank.
   - Else: `dist.recv(tensor, src=src)`. (No `copy_` needed here since we're receiving directly into the buffer we just built.)
4. Push the final tensor's values onto `out_queue` as a list, tagged with the rank: `out_queue.put((rank, tensor.tolist()))`.
5. `dist.destroy_process_group()`.

**Why every non-src rank ends up with the same data.** Send/recv is a point-to-point primitive; broadcasting from rank `src` means running `world_size - 1` matched pairs. The test confirms that all ranks see the original `payload` after the protocol runs.

In [ ]:
import os, datetime
import torch as t
import torch.distributed as dist

def ex1_broadcast_worker(rank, world_size, port, src, payload, out_queue):
    """Init gloo, broadcast `payload` from `src` via send/recv, queue result."""
    raise NotImplementedError()


def _test_ex1():
    import os as _os
    import datetime as _dt
    import torch.distributed as _dist
    import torch.multiprocessing as _mp

    def _dd_run_workers(worker_fn, world_size, port, *extra_args, timeout=30):
        """Spawn `world_size` fork-context procs, return list of exitcodes."""
        ctx = _mp.get_context('fork')
        procs = []
        for rank in range(world_size):
            p = ctx.Process(target=worker_fn, args=(rank, world_size, port, *extra_args))
            p.start()
            procs.append(p)
        for p in procs:
            p.join(timeout=timeout)
        codes = [p.exitcode for p in procs]
        for p in procs:
            if p.is_alive():
                p.terminate()
        return codes

    manager = _mp.Manager()
    q = manager.Queue()
    payload = [3.0, 1.0, 4.0, 1.0, 5.0]
    codes = _dd_run_workers(ex1_broadcast_worker, 3, 29530, 0, payload, q)
    assert codes == [0, 0, 0], f'workers failed: {codes}'

    # Drain queue.
    results = {}
    while not q.empty():
        rank, vals = q.get()
        results[rank] = vals
    assert set(results.keys()) == {0, 1, 2}, f'expected ranks 0,1,2, got {sorted(results.keys())}'
    for rank, vals in results.items():
        assert vals == payload, f'rank {rank} got {vals}, expected {payload}'

    # Test with src=1 (not rank 0).
    q2 = manager.Queue()
    payload2 = [7.0, 7.0, 7.0]
    codes2 = _dd_run_workers(ex1_broadcast_worker, 3, 29531, 1, payload2, q2)
    assert codes2 == [0, 0, 0], f'src=1 workers failed: {codes2}'
    results2 = {}
    while not q2.empty():
        rank, vals = q2.get()
        results2[rank] = vals
    for rank in [0, 1, 2]:
        assert results2[rank] == payload2, f'src=1: rank {rank} got {results2[rank]}'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_broadcast_worker(rank, world_size, port, src, payload, out_queue):
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = str(port)
    dist.init_process_group(backend='gloo', rank=rank, world_size=world_size,
                            timeout=datetime.timedelta(seconds=20))
    if rank == src:
        tensor = t.tensor(payload, dtype=t.float32)
        for other in range(world_size):
            if other != src:
                dist.send(tensor, dst=other)
    else:
        tensor = t.zeros(len(payload), dtype=t.float32)
        dist.recv(tensor, src=src)
    out_queue.put((rank, tensor.tolist()))
    dist.destroy_process_group()
```

**`send`/`recv` blocks**. Both are synchronous. If you accidentally have rank A wait to recv from rank B while rank B is waiting to recv from rank A, you get a deadlock and the process hangs forever. The 30s timeout in the test harness will eventually kill it, but in production you'd want `dist.isend` / `dist.irecv` (non-blocking) or — better — a higher-level collective like `dist.broadcast`.

**Why we don't use the real `dist.broadcast`.** `dist.broadcast` does exactly what this drill builds — in fewer lines and with NCCL tree-reduction acceleration. ARENA reimplements it by hand because the point is to internalize the protocol. In real code, always call `dist.broadcast(tensor, src=0)` and let the backend optimize.

**`zeros_like` vs `zeros`.** ARENA's solution uses `t.zeros_like(t)` where `t` is a freshly-constructed sender-side template. Here we use `t.zeros(len(payload), dtype=t.float32)` because the non-src rank doesn't have access to the payload — only its length and dtype, which must be agreed in advance. Same idea, slightly different ergonomics.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()